# ELT-02 · Carga bruta

- **Etapa:** ELT-02, carga bruta
- **Responsável:** Davi Mello (tarefa do Pedro, assumida pelo Davi)
- **Issue:** ELT-02
- **Entrada:** os 36 CSVs em `data/pessoas_*.csv`
- **Saída:** a tabela `raw.pessoas` no PostgreSQL

No ELT o dado vai **cru** para o banco e toda a limpeza acontece depois, em SQL. Por isso aqui o Python só envia os arquivos: quem lê o CSV é o próprio PostgreSQL, com `COPY` em `latin-1` e separador `;`. Todas as 31 colunas ficam como texto, do jeito que estão nos arquivos, e cada linha ganha `ano_origem` e `arquivo_origem`.

## 1. Setup

In [ ]:
import os
import time
from pathlib import Path

import pandas as pd
from sqlalchemy import text

# Raiz do repositório: sobe de pasta em pasta até achar a pasta .git
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
os.chdir(RAIZ)   # roda a partir da raiz, para achar a pasta src/ e o arquivo .env

from src.conexao import obter_engine, testar_conexao

DADOS    = RAIZ / "data"          # os 36 CSVs (estão no Git)
TABELA   = "raw.pessoas"          # destino desta etapa
ESPERADO = 806_977                # linhas dos 36 arquivos

testar_conexao()
engine = obter_engine()

def consulta(sql: str) -> pd.DataFrame:
    """Roda um SELECT e devolve uma tabela (text() deixa usar % do LIKE sem confusão)."""
    return pd.read_sql(text(sql), engine)

In [ ]:
# o schema raw é criado pela ELT-01 (sql/elt/00_schemas.sql)
with engine.connect() as conexao:
    existe_raw = conexao.execute(text("SELECT to_regnamespace('raw') IS NOT NULL")).scalar()
assert existe_raw, "o schema raw não existe: rode antes o sql/elt/00_schemas.sql (ELT-01)"
print("schema raw ok")

## 2. Conferir os arquivos antes de enviar

O Python só lê o cabeçalho e conta as linhas de cada arquivo, sem abrir os valores. Essas contagens são a referência para conferir a carga.

In [ ]:
arquivos = sorted(DADOS.glob("pessoas_*.csv"))
assert len(arquivos) == 36, f"esperava 36 arquivos, achei {len(arquivos)}"

def ler_cabecalho(arquivo: Path) -> list:
    """Nomes das colunas, lidos da primeira linha do arquivo."""
    with open(arquivo, encoding="latin-1") as f:
        return f.readline().strip().split(";")

def contar_linhas(arquivo: Path) -> int:
    """Linhas de dados (sem o cabeçalho). Aceita fim de linha do Windows ou do Linux."""
    return len(arquivo.read_bytes().splitlines()) - 1

COLUNAS = ler_cabecalho(arquivos[0])
diferentes = [a.name for a in arquivos if ler_cabecalho(a) != COLUNAS]
assert not diferentes, f"arquivos com colunas diferentes: {diferentes}"

no_csv = pd.DataFrame({
    "arquivo_origem": [a.name for a in arquivos],
    "ano_origem": [int(a.stem[-4:]) for a in arquivos],
    "linhas_csv": [contar_linhas(a) for a in arquivos],
})
print(len(COLUNAS), "colunas iguais nos 36 arquivos |", no_csv["linhas_csv"].sum(), "linhas")
assert no_csv["linhas_csv"].sum() == ESPERADO
no_csv.groupby("ano_origem")["linhas_csv"].agg(["count", "sum"]).rename(columns={"count": "arquivos", "sum": "linhas"})

## 3. Criar a tabela e carregar

A tabela é apagada e recriada a cada execução, dentro de **uma transação só**. Assim:
- rodar de novo não duplica nada;
- se algum arquivo falhar no meio, o banco volta ao estado anterior, sem carga pela metade.

Cada arquivo entra primeiro numa tabela temporária e depois vai para `raw.pessoas` junto com `ano_origem` e `arquivo_origem`. A tabela temporária some sozinha no fim da transação.

In [ ]:
DDL = (
    f"DROP TABLE IF EXISTS {TABELA};\n"
    f"CREATE TABLE {TABELA} (\n"
    + "".join(f"    {c} TEXT,\n" for c in COLUNAS)
    + "    ano_origem SMALLINT NOT NULL,\n"
    + "    arquivo_origem TEXT NOT NULL\n);\n"
    + f"COMMENT ON TABLE {TABELA} IS 'Pessoas envolvidas em sinistros (SP, 2021-2023): 36 CSVs crus, sem tratamento (ELT-02)';"
)
print(DDL)

In [ ]:
# o PostgreSQL lê o CSV: separador ;, primeira linha é cabeçalho, arquivo em latin-1
COPY_CSV = "COPY carga FROM STDIN WITH (FORMAT csv, DELIMITER ';', HEADER true, ENCODING 'LATIN1')"

def carregar(arquivos: list) -> float:
    """Recria raw.pessoas e carrega os arquivos numa transação só. Devolve o tempo em segundos."""
    inicio = time.perf_counter()
    conexao = engine.raw_connection()
    try:
        with conexao.cursor() as cur:
            cur.execute(DDL)
            colunas_texto = ", ".join(f"{c} TEXT" for c in COLUNAS)
            cur.execute(f"CREATE TEMP TABLE carga ({colunas_texto}) ON COMMIT DROP")
            for arquivo in arquivos:
                cur.execute("TRUNCATE carga")
                with open(arquivo, "rb") as f:
                    cur.copy_expert(COPY_CSV, f)
                cur.execute(f"INSERT INTO {TABELA} SELECT carga.*, %s, %s FROM carga",
                            (int(arquivo.stem[-4:]), arquivo.name))
            cur.execute(f"ANALYZE {TABELA}")   # estatísticas para as consultas das próximas etapas
        conexao.commit()
    except Exception:
        conexao.rollback()   # nada fica pela metade
        raise
    finally:
        conexao.close()
    return time.perf_counter() - inicio

tempo_carga = carregar(arquivos)
print(f"carga concluída em {tempo_carga:.1f} s")

## 4. Conferências

Seis conferências: linhas por arquivo, total por ano, chave da fonte, tipos das colunas, acentos e "nada foi alterado". Os números de referência da última conferência vêm do dicionário da fonte (ETL-11).

In [ ]:
# 1. linhas por arquivo: banco = CSV, nos 36 arquivos
no_banco = consulta(
    f"SELECT arquivo_origem, ano_origem, COUNT(*) AS linhas_banco FROM {TABELA} "
    "GROUP BY arquivo_origem, ano_origem")
por_arquivo = no_csv.merge(no_banco, on=["arquivo_origem", "ano_origem"], how="outer")
por_arquivo["confere"] = por_arquivo["linhas_csv"] == por_arquivo["linhas_banco"]
assert por_arquivo["confere"].all() and len(por_arquivo) == 36
print("os 36 arquivos conferem | total no banco:", int(por_arquivo["linhas_banco"].sum()))
por_arquivo.head(3)

In [ ]:
# 2. total por ano (fonte: 260.886 / 266.128 / 279.963)
por_ano = consulta(
    f"SELECT ano_origem, COUNT(*) AS linhas, COUNT(DISTINCT id_sinistro) AS sinistros "
    f"FROM {TABELA} GROUP BY ano_origem ORDER BY ano_origem")
assert por_ano["linhas"].tolist() == [260_886, 266_128, 279_963]
por_ano

In [ ]:
# 3. a chave da fonte veio inteira: um id_pessoa por linha, nenhum vazio
chave = consulta(
    f"SELECT COUNT(*) AS linhas, COUNT(DISTINCT id_pessoa) AS id_pessoa_distintos, "
    f"COUNT(*) - COUNT(id_pessoa) AS id_pessoa_vazios FROM {TABELA}")
assert chave.iloc[0].tolist() == [ESPERADO, ESPERADO, 0]
chave

In [ ]:
# 4. tipos: as 31 colunas da fonte são texto; só as 2 de rastreio foram acrescentadas
tipos = consulta(
    "SELECT data_type AS tipo, COUNT(*) AS colunas FROM information_schema.columns "
    "WHERE table_schema = 'raw' AND table_name = 'pessoas' GROUP BY data_type")
assert dict(zip(tipos["tipo"], tipos["colunas"])) == {"text": 32, "smallint": 1}
tipos

In [ ]:
# 5. acentos: o latin-1 virou UTF-8 certo no banco (nenhum caractere de substituição)
acentos = consulta(f"""
    SELECT
        COUNT(*) FILTER (WHERE regiao_administrativa = 'METROPOLITANA DE SÃO PAULO') AS regiao_metropolitana_sp,
        COUNT(*) FILTER (WHERE regiao_administrativa = 'SÃO JOSÉ DO RIO PRETO')      AS regiao_sao_jose_rio_preto,
        COUNT(*) FILTER (WHERE grau_de_instrucao = 'MÉDIO')                         AS instrucao_medio,
        COUNT(*) FILTER (WHERE strpos(municipio || regiao_administrativa
                                      || COALESCE(grau_de_instrucao, '')
                                      || COALESCE(profissao, ''), chr(65533)) > 0) AS caracteres_invalidos
    FROM {TABELA}""")
assert (acentos.iloc[0, :3] > 0).all() and acentos.loc[0, "caracteres_invalidos"] == 0
acentos.T.rename(columns={0: "linhas"})

In [ ]:
# 6. nada foi alterado: os "defeitos" da fonte continuam lá, para a limpeza em SQL tratar
cru = consulta(f"""
    SELECT
        COUNT(*) FILTER (WHERE nacionalidade LIKE '% ')                AS nacionalidade_com_espaco,
        COUNT(*) FILTER (WHERE tipo_de_vitima = 'NAO DISPONIVEL')      AS tipo_de_vitima_nao_disponivel,
        COUNT(*) FILTER (WHERE gravidade_lesao = 'NAO DISPONIVEL')     AS gravidade_nao_disponivel,
        COUNT(*) - COUNT(id_veiculo)                                   AS id_veiculo_vazio,
        COUNT(*) - COUNT(data_obito)                                   AS data_obito_vazia,
        COUNT(*) FILTER (WHERE municipio LIKE '% D OESTE')             AS municipio_apostrofo_perdido
    FROM {TABELA}""")
REFERENCIA = {   # dicionário da fonte (ETL-11)
    "nacionalidade_com_espaco": 711_291, "tipo_de_vitima_nao_disponivel": 112_828,
    "gravidade_nao_disponivel": 70_314, "id_veiculo_vazio": 169_754, "data_obito_vazia": 791_184,
}
assert all(cru.loc[0, k] == v for k, v in REFERENCIA.items())
cru.T.rename(columns={0: "linhas"})

## 5. Rodar de novo não duplica

A carga roda uma segunda vez e a contagem tem que continuar 806.977.

In [ ]:
tempo_recarga = carregar(arquivos)
with engine.connect() as conexao:
    depois_da_recarga = conexao.execute(text(f"SELECT COUNT(*) FROM {TABELA}")).scalar()
assert depois_da_recarga == ESPERADO
print(f"segunda carga: {depois_da_recarga} linhas (sem duplicar) em {tempo_recarga:.1f} s")

## 6. Tamanho do banco

A fase ELT guarda no mesmo banco os dados crus, a limpeza e a segunda estrela. Por isso medimos o espaço desde já.

In [ ]:
tamanhos = consulta("""
    SELECT n.nspname AS schema,
           COUNT(*) FILTER (WHERE c.relkind = 'r')          AS tabelas,
           ROUND(SUM(pg_total_relation_size(c.oid)) / 1024.0 / 1024, 1) AS mb
    FROM pg_class c
    JOIN pg_namespace n ON n.oid = c.relnamespace
    WHERE c.relkind IN ('r', 'm') AND n.nspname IN ('public', 'raw', 'stg', 'dw_elt')
    GROUP BY n.nspname ORDER BY n.nspname""")

with engine.connect() as conexao:
    banco_mb = conexao.execute(text("SELECT pg_database_size(current_database()) / 1024.0 / 1024")).scalar()
csv_mb = sum(a.stat().st_size for a in arquivos) / 1024 / 1024

print(f"CSVs no disco: {csv_mb:.1f} MB | raw.pessoas: {tamanhos.set_index('schema').loc['raw', 'mb']} MB "
      f"| banco inteiro: {float(banco_mb):.1f} MB")
tamanhos

## 7. Resumo

**O que foi feito:** carga dos 36 CSVs mensais (2021 a 2023) na tabela `raw.pessoas`, sem nenhum tratamento. O PostgreSQL lê cada arquivo direto com `COPY` (`latin-1`, separador `;`). As 31 colunas da fonte ficam como texto, e cada linha ganha `ano_origem` e `arquivo_origem`.

**Por que assim:** no ELT a transformação acontece dentro do banco. Deixar tudo como texto garante que nenhum valor é perdido ou convertido na carga. A conversão de tipos, os nulos e o texto ficam para as etapas em SQL (ELT-03 em diante).

**Rodar mais de uma vez:** a tabela é recriada a cada execução, numa transação só. Rodar de novo não duplica, e uma falha no meio não deixa carga pela metade.

**O que foi conferido:**
- as 806.977 linhas, arquivo por arquivo, iguais às dos CSVs, e 260.886, 266.128 e 279.963 por ano;
- `id_pessoa` único e sem vazios;
- as 31 colunas da fonte em texto;
- acentos corretos (`METROPOLITANA DE SÃO PAULO`, `MÉDIO`), sem caracteres inválidos;
- os "defeitos" da fonte continuam lá (espaços em `nacionalidade`, `NAO DISPONIVEL`, campos vazios como nulo), iguais ao dicionário da fonte (ETL-11);
- a segunda execução mantém 806.977 linhas.

**Tamanho:** a seção 6 mostra quanto `raw.pessoas` ocupa no banco, comparado aos CSVs e às outras camadas. O tempo da carga (seção 3) entra no comparativo ETL × ELT (ELT-10).

**Próximo passo:** a ELT-03 lê `raw.pessoas` e faz a primeira limpeza em SQL no schema `stg`.